# Implementing Neural Networks
## Step 1: Set up an Azure Machine Learning workspace
Before implementing a neural network, we need to set up an Azure Machine Learning workspace. You can do this by following these steps:
Install the Azure Machine Learning SDK
Next, create a new workspace or use an existing one:

In [ ]:
# Install the Azure Machine Learning SDK
# pip install azureml-core

from azureml.core import Workspace
from azureml.core.authentication import AzureCliAuthentication

cli_auth = AzureCliAuthentication()

# Create or retrieve an existing Azure ML workspace
ws = Workspace.get(name='kafka-team-azure-ml-ws',
                   subscription_id='4517e28e-a71e-4f99-bef7-312286beb95e',
                   resource_group='ddit-lcm-kafka-westeurope-rg',
                   auth=cli_auth)

print(ws)

# Write configuration to the workspace config file
ws.write_config(path='.azureml')

Workspace.create(name='kafka-team-azure-ml-ws', subscription_id='4517e28e-a71e-4f99-bef7-312286beb95e', resource_group='ddit-lcm-kafka-westeurope-rg')


# Step 2: Build a simple neural network using PyTorch
Now that we have our workspace set up, let’s define a simple neural network. PyTorch is well-suited for building neural networks, and it integrates seamlessly with Azure.

Here's a basic implementation of a feedforward neural network:

In the below code, we:
* Defined a simple neural network with an input layer, one hidden layer, and an output layer.
* Used the ReLU (rectified linear unit) activation function, which is commonly used in neural networks.
* Defined a CrossEntropyLoss for multi-class classification and SGD (stochastic gradient descent) as the optimizer.



In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define a simple neural network with one hidden layer
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(784, 128)  # Input layer (784 input features)
        self.fc2 = nn.Linear(128, 10)   # Output layer (10 classes)
        self.relu = nn.ReLU()           # Activation function

    def forward(self, x):
        x = self.relu(self.fc1(x))  # Apply ReLU after the first layer
        x = self.fc2(x)             # Output layer
        return x

# Initialize the neural network
model = SimpleNN()

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # For classification tasks
optimizer = optim.SGD(model.parameters(), lr=0.01)  # Stochastic Gradient Descent

## Step 3: Train the neural network
After defining the model, we need to train it on a dataset. We will simulate training using the MNIST dataset (a set of handwritten digits) to demonstrate the process. First, we load the data and then train the model.

In this code:
* The MNIST dataset is loaded using torchvision.
* The network is trained for five epochs using stochastic gradient descent to update the weights.
* The input is flattened (from 28 × 28 images to 784 features) before being fed into the network.

In [5]:
from torchvision import datasets, transforms

# Load MNIST dataset
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(root='data', train=True, download=True,
                   transform=transforms.ToTensor()),
    batch_size=32, shuffle=True)

# Train the model
num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()  # Reset gradients

        # Forward pass
        outputs = model(inputs.view(-1, 784))  # Flatten input
        loss = criterion(outputs, labels)      # Compute loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

100.0%
100.0%
100.0%
100.0%


Epoch 1, Loss: 0.8617986896753311
Epoch 2, Loss: 0.37486770164569216
Epoch 3, Loss: 0.3216451079249382
Epoch 4, Loss: 0.2921473574638367
Epoch 5, Loss: 0.2686258118212223


## Step 4: Deploy the model on Azure
After training the model, you can deploy it to Azure Machine Learning for inference at scale. Here's how to register the model and deploy it as a web service:

In [17]:
from azureml.core import Model

# Save the trained model
torch.save(model.state_dict(), 'simple_nn.pth')

# Register the model in Azure (with error handling)
try:
	registered_model = Model.register(workspace=ws, model_path='simple_nn.pth', model_name='simple_nn')
	print(f"Model registered successfully: {registered_model.name}")
except Exception as e:
	print(f"Authorization error: {e}")
	print("Ensure you have the correct Azure credentials and permissions to register models in this workspace.")

# Deploying as a web service requires more steps such as creating a scoring script and environment.

Authorization error: This request is not authorized to perform this operation.
RequestId:89400c1b-d01e-004b-6363-970b8e000000
Time:2026-02-06T12:25:52.6183329Z
ErrorCode:AuthorizationFailure
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>AuthorizationFailure</Code><Message>This request is not authorized to perform this operation.
RequestId:89400c1b-d01e-004b-6363-970b8e000000
Time:2026-02-06T12:25:52.6183329Z</Message></Error>
Ensure you have the correct Azure credentials and permissions to register models in this workspace.


At this point, the model can be registered and further steps can be taken to deploy it as an Azure web service.

## Guide summary
* Workspace setup: Using Azure ML SDK to create or retrieve a workspace.
* Model building: Creating a simple feedforward network with PyTorch, defining the architecture and activation function.
* Training: Using PyTorch’s data loader and optimizer to train the neural network with backpropagation.
* Deployment: Registering the model for deployment on Azure.